# e1-esm-benchmark — Kaggle / Colab runner

Upload this notebook to **Kaggle** (preferred — longer sessions, easier persistent cache) or **Google Colab**, enable a GPU, then **Run All**.

- **Kaggle**: Settings → Accelerator → *GPU T4 x2*. Save outputs to a Kaggle Dataset to reuse the embedding cache across sessions.
- **Colab**: Runtime → Change runtime type → *T4 GPU*. Mount Google Drive (cell below) to persist the cache.
- **Local**: just open this in Jupyter from the repo root and Run All — no clone or Drive mount needed.

What this notebook does:
1. Detects the environment (Kaggle / Colab / local).
2. Clones the repo if missing; installs deps (`pip install -e .`) — first install is slow because `E1` builds from git.
3. Points `cache/`, `data/`, `results/` at persistent storage so re-runs don't re-embed.
4. Runs the chosen task and shows the results table inline.

## Parameters — edit these

In [ ]:
# EDIT to your repo URL (only used on Kaggle / Colab when no local clone is present).
GITHUB_REPO = "https://github.com/boheling/e1-esm-benchmark"

# Task: meltome | gb1 | fluorescence  (proteingym has its own cell at the end)
TASK = "fluorescence"

# Heads to sweep and encoders to compare
HEADS = ["ridge", "lasso", "mlp"]
ENCODERS = ["esm2", "e1"]

## 1. Environment + persistent paths

In [ ]:
import os, sys, pathlib

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENV = "kaggle"
    WORK = pathlib.Path("/kaggle/working")
    PERSIST = WORK  # /kaggle/working is preserved on commit; save as Dataset for cross-session reuse
elif "COLAB_GPU" in os.environ or "google.colab" in sys.modules:
    ENV = "colab"
    WORK = pathlib.Path("/content")
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        PERSIST = pathlib.Path("/content/drive/MyDrive/e1-esm-benchmark")
        PERSIST.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted -> persistent storage: {PERSIST}")
    except Exception as e:
        print(f"Drive mount skipped ({e}); using ephemeral /content for cache.")
        PERSIST = WORK
else:
    ENV = "local"
    WORK = pathlib.Path.cwd()
    PERSIST = WORK

print(f"Environment: {ENV}")
print(f"Working dir: {WORK}")
print(f"Persistent cache root: {PERSIST}")

## 2. Clone repo (Kaggle/Colab only) and install

In [ ]:
import subprocess

# If we're already inside the repo (e.g. running locally), just use that.
_here = pathlib.Path.cwd()
if (_here / "pyproject.toml").exists() and (_here / "src/e1_esm_benchmark").exists():
    REPO_DIR = _here
    print(f"Using existing repo at {REPO_DIR}")
else:
    REPO_DIR = WORK / "e1-esm-benchmark"
    if not REPO_DIR.exists():
        print(f"Cloning {GITHUB_REPO} -> {REPO_DIR}")
        subprocess.check_call(["git", "clone", "--depth", "1", GITHUB_REPO, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print(f"Working in {REPO_DIR}")

In [ ]:
# Install. First run takes ~5 min because E1 is built from git.
# Subsequent runs are seconds (cached wheel).
%pip install --quiet -e .
print("Installed.")

## 3. Wire `cache/`, `data/`, `results/` to persistent storage

If you re-run this notebook in a fresh session, embeddings under `cache/embeddings/` will be re-used — that's the slow step.

In [ ]:
for sub in ["cache", "data", "results"]:
    persistent = PERSIST / sub
    persistent.mkdir(parents=True, exist_ok=True)
    link = REPO_DIR / sub
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        # Existing dir from a prior local run — leave it alone
        print(f"  {sub}: keeping existing local dir {link}")
        continue
    link.symlink_to(persistent)
    print(f"  {sub}: {link} -> {persistent}")

## 4. GPU sanity check

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"CUDA OK — device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no CUDA. Embedding 600M-param models on CPU is impractical — enable the GPU runtime before continuing.")

## 5. Run the chosen task

First pass embeds the data, writes `.npy` caches under `cache/embeddings/`, fits the head, and appends a row to `results/benchmark.csv`. Subsequent passes for new heads are seconds — the embeddings are cached.

**Estimates on a T4** (free Kaggle / Colab tier):
- `fluorescence` full sweep: ~1.5 h
- `gb1` full sweep: ~3 h
- `meltome` full sweep: ~4 h

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

from e1_esm_benchmark.benchmark import run_one, append_results

for enc in ENCODERS:
    for head in HEADS:
        row = run_one(task=TASK, encoder=enc, head=head)
        append_results(row)

## 6. Render the results table

In [ ]:
import datetime, glob
from IPython.display import Markdown, display

today = datetime.date.today().isoformat()
md_files = sorted(glob.glob(f"results/comparison-*{TASK}*-{today}.md"))
if md_files:
    display(Markdown(pathlib.Path(md_files[-1]).read_text()))
else:
    print("No results markdown produced yet — did the run cell finish?")

import pandas as pd
csv = pathlib.Path("results/benchmark.csv")
if csv.exists():
    print("\nFull benchmark.csv (last 12 rows):")
    display(pd.read_csv(csv).tail(12))

## 7. (Optional) ProteinGym sweep

Multi-assay aggregation. **Start small** — full 217 is many hours on a T4.

First run downloads the ProteinGym DMS_substitutions config (~2 GB) into `data/proteingym/`. On Kaggle, save that to a Dataset to reuse later.

In [ ]:
PG_N_ASSAYS = 5   # bump to 10–20 if you have time; 0 = all 217 (slow)

from e1_esm_benchmark.benchmark_proteingym import run_proteingym

for enc in ENCODERS:
    for head in ["ridge"]:   # add lasso/mlp here if desired
        run_proteingym(
            encoder=enc, head=head,
            n_assays=None if PG_N_ASSAYS == 0 else PG_N_ASSAYS,
        )

In [ ]:
# Per-assay drilldown
import pandas as pd, glob
for f in sorted(glob.glob(f"results/proteingym-per-assay-*-{today}.csv")):
    print(f"\n{f}")
    display(pd.read_csv(f).round(3))